# E0 · Re-treinar o twitter-XLM-R e guardar os pesos

O painel neural de 24/08 mediu este modelo e ele ganhou do stack servido em
**todos** os eixos do teste v5 (16.261 linhas):

| | macro-F1 | recall ódio | precisão |
|---|---|---|---|
| stack servido | 0,7906 | 0,7359 | 0,6812 |
| **twitter-XLM-R** | **0,8477** | **0,7707** | **0,7945** |
| stack, fatia PT | 0,6966 | 0,3162 | 0,6789 |
| **twitter-XLM-R, fatia PT** | **0,7646** | **0,5074** | 0,6389 |

O problema é que **os pesos não existem mais**: o notebook do painel salvava
métricas e predições, não o modelo. Este aqui re-treina com a mesma receita e
**guarda os pesos no Drive**, para o notebook 2 exportar e medir sem GPU.

**Runtime: GPU (T4).** Leva algo entre 40 e 90 minutos.

No fim ele baixa `xlmr_treino_resultado.zip`. É esse arquivo que eu preciso.

## 1 · GPU

In [ ]:
!nvidia-smi -L

## 2 · Dependências, repositório e imports (célula única)

In [ ]:
# only-if-needed: o Colab ja traz transformers, torch e accelerate. Forcar upgrade
# aqui rebaixa o huggingface-hub e quebra pacotes pre-instalados que dependem dele.
!pip install -q --upgrade-strategy only-if-needed 'transformers>=4.44' 'datasets>=2.18' \
    'accelerate>=0.29' 'sentencepiece>=0.2' 'scikit-learn>=1.4' 'pyarrow>=15' 'pyyaml>=6.0'

!git clone -q https://github.com/isasaade-23/hate-speech-nlp-en-pt.git /content/repo

import inspect, json, os, shutil, sys, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from datasets import Dataset
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

REPO = Path('/content/repo')
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('repo em', REPO, '| commit', os.popen('git rev-parse --short HEAD').read().strip())
print('torch', torch.__version__, '| GPU', torch.cuda.is_available())

## 3 · O corpus

O corpus **não está no git de propósito**: ele mistura HateBR, que é
research-only e não pode ser redistribuído num repositório público. Então ele
vem do Drive.

Coloque `corpus_strict.parquet` (de `data/processed/` no seu PC) em
`Meu Drive/luciola/` antes de rodar. Se não estiver lá, a célula abre o widget
de upload como alternativa.

A trava anti-mojibake é a mesma do painel v5: ela recusa corpus com português
corrompido, porque isso já aconteceu uma vez e passou despercebido.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/luciola'
CORPUS = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    p = Path(DRIVE_DIR) / 'corpus_strict.parquet'
    if p.exists():
        CORPUS = pd.read_parquet(p)
        print('corpus do Drive:', p)
except ImportError:
    pass

if CORPUS is None:
    from google.colab import files
    print('nao achei no Drive. Suba o corpus_strict.parquet:')
    up = files.upload()
    CORPUS = pd.read_parquet(list(up)[0])

# trava anti-mojibake: PT correto tem acento; 'Ã©' e 'Ã§' sao sinal de UTF-8 lido como latin-1
amostra = ' '.join(CORPUS[CORPUS.language == 'pt'].text_clean.head(2000).astype(str))
assert 'Ã©' not in amostra and 'Ã§' not in amostra, 'corpus com mojibake, refaca o export'
assert any(c in amostra for c in 'áéíóúçãõ'), 'nenhum acento na fatia PT, corpus suspeito'

print(CORPUS.groupby(['split', 'language']).size().unstack(fill_value=0))
print('total', len(CORPUS), '| odio', int(CORPUS.label.sum()))

## 4 · A receita, lida de `configs/neural/twitter_xlmr.yaml`

Nada de hiperparâmetro digitado à mão aqui: se o arquivo do repositório mudar,
este notebook muda junto, e a comparação com o painel de agosto continua de pé.

In [ ]:
CFG_YAML = yaml.safe_load(open('configs/neural/twitter_xlmr.yaml', encoding='utf-8'))
CFG = {
    'ckpt': CFG_YAML['model']['hf_checkpoint'],
    'max_length': CFG_YAML['model']['max_length'],
    'epochs': CFG_YAML['train']['epochs'],
    'batch_size': CFG_YAML['train']['batch_size'],
    'lr': float(CFG_YAML['train']['lr']),
    'weight_decay': CFG_YAML['train']['weight_decay'],
    'warmup_ratio': CFG_YAML['train']['warmup_ratio'],
    'languages': CFG_YAML['languages'],
}
TEXT_COL = CFG_YAML['text_column']
POLICY, SEED = CFG_YAML['policy'], CFG_YAML['seed']
MID = f"twitter_xlmr_{POLICY}_s{SEED}"
for k, v in CFG.items():
    print(f'{k:14s} {v}')
print(f'{"text_column":14s} {TEXT_COL}\n{"model_id":14s} {MID}')

## 5 · Métricas e limiar, idênticos ao `hsc`

Copiados do painel v5 sem alteração. O limiar é ajustado **na validação**,
varrendo quantis e maximizando macro-F1, e o teste é tocado uma vez só.

In [ ]:
POS = 1

def classification_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    out = {'n': int(len(y_true)), 'n_hate': int((y_true == POS).sum()),
           'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
           'accuracy': float((y_true == y_pred).mean()),
           'precision_hate': float(p[1]), 'recall_hate': float(r[1]), 'f1_hate': float(f[1]),
           'precision_nothate': float(p[0]), 'recall_nothate': float(r[0])}
    if y_score is not None and len(np.unique(y_true)) == 2:
        y_score = np.asarray(y_score)
        out['roc_auc'] = float(roc_auc_score(y_true, y_score))
        out['pr_auc'] = float(average_precision_score(y_true, y_score))
    return out

def best_threshold(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score, dtype=float)
    cands = np.unique(np.quantile(y_score, np.linspace(0.02, 0.98, 97)))
    best_t, best_f = 0.5, -1.0
    for t in cands:
        f = f1_score(y_true, (y_score >= t).astype(int), average='macro', zero_division=0)
        if f > best_f: best_f, best_t = f, float(t)
    return best_t

def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=1000, seed=42, alpha=0.05):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed); n = len(y_true); stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[b] = f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0)
    return float(np.percentile(stats, 100*alpha/2)), float(np.percentile(stats, 100*(1-alpha/2)))

def breakdown(df, by):
    rows = []
    for key, g in df.groupby(by):
        m = classification_metrics(g['label'], g['pred'])
        rows.append({by: key, 'n': m['n'], 'n_hate': m['n_hate'],
                     'macro_f1': round(m['macro_f1'], 4), 'recall_hate': round(m['recall_hate'], 4)})
    return sorted(rows, key=lambda r: str(r[by]))

def metrics_block(part, y_pred, y_score, seed):
    y = part['label'].values; y_pred = np.asarray(y_pred)
    m = classification_metrics(y, y_pred, y_score)
    lo, hi = bootstrap_macro_f1_ci(y, y_pred, seed=seed)
    m['macro_f1_ci95'] = [round(lo, 4), round(hi, 4)]
    m['confusion'] = confusion_matrix(y, y_pred, labels=[0, 1]).tolist()
    pdf = part[['language', 'source_dataset', 'label']].copy(); pdf['pred'] = y_pred
    m['by_language'] = breakdown(pdf, 'language'); m['by_source'] = breakdown(pdf, 'source_dataset')
    return m

def prediction_frame(part, y_score, threshold):
    y_score = np.asarray(y_score, dtype=float)
    return pd.DataFrame({'id': part['id'].values, 'language': part['language'].values,
                         'source_dataset': part['source_dataset'].values,
                         'y_true': part['label'].values.astype(int), 'y_score': y_score,
                         'y_pred': (y_score >= threshold).astype(int)})
print('helpers definidos')

## 6 · Treinar

A diferença para o painel de agosto está em duas linhas no fim: **salvar os
pesos e o tokenizer**. Foi a ausência disso que obrigou a re-treinar agora.

In [ ]:
os.makedirs('reports/metrics', exist_ok=True)
os.makedirs('reports/predictions', exist_ok=True)
PESOS = Path(DRIVE_DIR) / 'modelos' / MID

torch.manual_seed(SEED); np.random.seed(SEED)
df = CORPUS[CORPUS.language.isin(CFG['languages'])]
tr, va, te = (df[df.split == s] for s in ('train', 'val', 'test'))
print(f'train={len(tr)} val={len(va)} test={len(te)}')

tok = AutoTokenizer.from_pretrained(CFG['ckpt'])
def enc(b): return tok(b[TEXT_COL], truncation=True, max_length=CFG['max_length'])
def to_ds(p):
    d = Dataset.from_pandas(p[[TEXT_COL, 'label']].reset_index(drop=True))
    return d.map(enc, batched=True).rename_column('label', 'labels')
ds_tr, ds_va, ds_te = to_ds(tr), to_ds(va), to_ds(te)

model = AutoModelForSequenceClassification.from_pretrained(CFG['ckpt'], num_labels=2)
cnt = tr['label'].value_counts().to_dict(); n = len(tr)
w = torch.tensor([n/(2*cnt.get(0, 1)), n/(2*cnt.get(1, 1))], dtype=torch.float)
print('class weights', w.tolist())

class WT(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        lab = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=w.to(out.logits.device))(out.logits, lab)
        return (loss, out) if return_outputs else loss

ta = dict(output_dir=f'/content/hf/{MID}', num_train_epochs=CFG['epochs'],
          per_device_train_batch_size=CFG['batch_size'], per_device_eval_batch_size=32,
          learning_rate=CFG['lr'], weight_decay=CFG['weight_decay'],
          fp16=torch.cuda.is_available(), eval_strategy='epoch', save_strategy='no',
          logging_steps=50, seed=SEED, report_to=[])
# 10% de warmup nas duas APIs: transformers <5 tem warmup_ratio; v5 so warmup_steps
if 'warmup_ratio' in inspect.signature(TrainingArguments.__init__).parameters:
    ta['warmup_ratio'] = CFG['warmup_ratio']
else:
    passos = (len(ds_tr) + CFG['batch_size'] - 1) // CFG['batch_size']
    ta['warmup_steps'] = max(1, int(CFG['warmup_ratio'] * passos * CFG['epochs']))

trainer = WT(model=model, args=TrainingArguments(**ta), train_dataset=ds_tr,
             eval_dataset=ds_va, data_collator=DataCollatorWithPadding(tok))
trainer.train()

def sc(d): return torch.softmax(torch.tensor(trainer.predict(d).predictions), 1).numpy()[:, 1]
val_s, test_s = sc(ds_va), sc(ds_te)
THR = best_threshold(va['label'].values, val_s)

res = {'model_id': MID, 'config': 'twitter_xlmr', 'family': 'neural', 'policy': POLICY,
       'seed': SEED, 'hf_checkpoint': CFG['ckpt'], 'n_train': int(len(tr)),
       'threshold': round(float(THR), 4), 'splits': {}}
for split, part, sco in (('val', va, val_s), ('test', te, test_s)):
    yp = (np.asarray(sco) >= THR).astype(int)
    res['splits'][split] = metrics_block(part, yp, sco, SEED)
    prediction_frame(part, sco, THR).to_parquet(f'reports/predictions/{MID}_{split}.parquet', index=False)
json.dump(res, open(f'reports/metrics/{MID}.json', 'w'), indent=2)

# O QUE FALTAVA NO PAINEL DE AGOSTO: guardar o modelo.
PESOS.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(PESOS)); tok.save_pretrained(str(PESOS))
json.dump({'model_id': MID, 'threshold': round(float(THR), 4), 'hf_checkpoint': CFG['ckpt'],
           'max_length': CFG['max_length'], 'text_column': TEXT_COL},
          open(PESOS / 'luciola_serve.json', 'w'), indent=2)

t = res['splits']['test']
print(f"\nlimiar {THR:.4f} | teste macro-F1 {t['macro_f1']:.4f} recall {t['recall_hate']:.4f}")
print('pesos em', PESOS)

## 7 · Trava de reprodução

O repositório clonado já traz `reports/metrics/twitter_xlmr_strict_s42.json` da
corrida de agosto. Se esta corrida não bater com aquela dentro de uma margem, o
problema é aqui e não adianta seguir para o notebook 2.

Margem de 0,02 em macro-F1 e 0,04 em recall: seed fixa não garante bit a bit em
GPU diferente, mas variação maior que isso quer dizer receita diferente.

In [ ]:
ref = json.load(open('/content/repo/reports/metrics/twitter_xlmr_strict_s42.json'))['splits']['test']
novo = res['splits']['test']

linhas, ok = [], True
for campo, margem in (('macro_f1', 0.02), ('recall_hate', 0.04), ('precision_hate', 0.04)):
    d = abs(novo[campo] - ref[campo]); passou = d <= margem; ok &= passou
    linhas.append({'metrica': campo, 'agosto': round(ref[campo], 4),
                   'agora': round(novo[campo], 4), 'delta': round(d, 4),
                   'margem': margem, 'passou': passou})

por_lingua = {r['language']: r for r in novo['by_language']}
ref_lingua = {r['language']: r for r in ref['by_language']}
print(pd.DataFrame(linhas).to_string(index=False))
print()
for lg in ('en', 'pt'):
    a, b = ref_lingua.get(lg, {}), por_lingua.get(lg, {})
    print(f"{lg}: recall agosto {a.get('recall_hate')} -> agora {b.get('recall_hate')}"
          f" | macro-F1 {a.get('macro_f1')} -> {b.get('macro_f1')}")

print('\n' + ('REPRODUZIU' if ok else '*** NAO REPRODUZIU: pare aqui e me mande a saida ***'))

## 8 · Empacotar

O zip traz métricas, predições e o registry. Os pesos ficam no Drive, porque
1,1 GB não passa por download de notebook.

In [ ]:
REGISTRY = {MID: {'path': f'models/{MID}/hf', 'config': 'twitter_xlmr', 'family': 'neural',
                  'policy': POLICY, 'seed': SEED, 'hf_checkpoint': CFG['ckpt'],
                  'threshold': round(float(THR), 4),
                  'val_macro_f1': res['splits']['val']['macro_f1'],
                  'test_macro_f1': res['splits']['test']['macro_f1']}}
json.dump(REGISTRY, open('registry_neural.json', 'w'), indent=2)

ALVO = '/content/xlmr_treino_resultado.zip'
with zipfile.ZipFile(ALVO, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(f'reports/metrics/{MID}.json', f'reports/metrics/{MID}.json')
    for split in ('val', 'test'):
        f = f'reports/predictions/{MID}_{split}.parquet'
        z.write(f, f)
    z.write('registry_neural.json', 'models/registry_neural.json')
shutil.copy(ALVO, Path(DRIVE_DIR) / 'xlmr_treino_resultado.zip')
print('no Drive tambem:', Path(DRIVE_DIR) / 'xlmr_treino_resultado.zip')

try:
    from google.colab import files
    files.download(ALVO)
except ImportError:
    print('rodando local:', ALVO)

---

**Me mande:** o `xlmr_treino_resultado.zip` e a saída da célula 7 (a trava de
reprodução). Se a trava reprovar, me mande só a saída dela e pare: não vale
gastar o notebook 2 em cima de um modelo que não reproduziu.

Confirme também que `Meu Drive/luciola/modelos/twitter_xlmr_strict_s42/` existe
e tem uns 1,1 GB. É de lá que o notebook 2 parte.